# 0_get_basin_groups.ipynb

This notebook is the very first step in preprocessing. Its job is to pair up all the locations with gauge data with their MERIT networks. This allows us to build the csv file of gauge locations and basin groups used in the basin delineation done next.

In [30]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

target_crs = "EPSG:4326"

datasets = Path("/nas/cee-ice/data")

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/")
metadata_dir = save_dir / "metadata" 
metadata_dir.mkdir(exist_ok=True)

In [2]:
import global_gauges as gg
facade = gg.GaugeDataFacade(providers=['usgs','eccc','brana','ukea','eauf','abom'])
sites = facade.get_stations_n_days(365*10).to_crs(target_crs)

/nas/cee-water/cjgleason/ted/global_gauges/global_gauges/facade.py:213: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(provider_info)


In [3]:
csv_files = list(Path('/nas/cee-water/cjgleason/ted/gauge-match/data').glob('*.csv'))
matches = pd.concat([pd.read_csv(f)[['site_id','COMID']] for f in csv_files]) # dont need other data
matches['COMID'] = pd.to_numeric(matches['COMID'], errors='coerce')
matches.dropna(inplace=True)

matches

,site_id,COMID
0,BRANA-12160000,62054100.0
1,BRANA-12420000,62070024.0
2,BRANA-14380000,62011372.0
3,BRANA-14489000,62000288.0
5,BRANA-15040000,62048834.0
...,...,...
11109,USGS-02458600,73012694.0
11110,USGS-02460500,73012694.0
11111,USGS-09113500,77015788.0
11112,USGS-09113980,77015788.0


In [4]:
matched_sites = sites.merge(matches, left_index=True, right_on='site_id').reset_index(drop=True)
matched_sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider,site_id,COMID
0,Bradford On Avon,NaN,True,51.342781,-2.266665,2026-02-03 18:09:26.056908,2005-05-19,2026-02-02,0.422,212.344,11.605689,7563.0,{'guid': '585e49e0-7070-4d85-9c52-ccd18a5a1e65'},POINT (-2.26666 51.34278),ukea,UKEA-530176,23009540.0
1,Tavistock Abbey Bridge,NaN,True,50.550625,-4.142658,2026-02-03 18:15:23.871451,1991-07-25,2026-02-01,0.230,70.092,4.410262,12608.0,{'guid': '603fcc73-a9aa-4831-bf66-6a1a2b0a04d7'},POINT (-4.14266 50.55062),ukea,UKEA-SX47F022,23012184.0
2,Izaak Walton,NaN,True,53.056088,-1.781900,2026-02-03 18:13:27.803203,1969-07-01,2026-02-01,0.274,26.324,1.984309,20666.0,{'guid': '73ee60b4-ef40-4579-a060-db2312ea3ce1'},POINT (-1.7819 53.05609),ukea,UKEA-4046,23003236.0
3,Duddon Hall,NaN,True,54.295504,-3.237944,2026-02-03 18:06:24.406436,1976-04-01,2026-02-01,0.147,161.898,5.430623,18200.0,{'guid': 'bdd1ae70-1a37-4d43-9e04-bd378c327086'},POINT (-3.23794 54.2955),ukea,UKEA-740102,23004260.0
4,Stanhope,NaN,True,54.746300,-2.026897,2026-02-03 18:15:08.786619,1958-10-01,2026-02-01,0.000,138.506,3.978364,24592.0,{'guid': 'b29c481a-5012-40f5-bb0c-f9370be34975'},POINT (-2.0269 54.7463),ukea,UKEA-024003,23001202.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12040,Le Woigot à Briey,NaN,True,49.246953,5.933378,2026-02-03 18:41:00.753027,1967-06-15,2026-02-02,0.003,22.034,1.271736,21376.0,None,POINT (5.93338 49.24695),eauf,EAUF-A8322010,23015393.0
12041,La Lézarde à Montivilliers,NaN,True,49.543395,0.186444,2026-02-03 18:28:48.638475,2003-03-26,2026-02-02,0.590,7.824,1.080216,8349.0,None,POINT (0.18644 49.54339),eauf,EAUF-H7330410,23016883.0
12042,La Jonte à Meyrueis [aval],NaN,True,44.180086,3.428805,2026-02-03 18:08:48.100035,1969-09-18,2026-02-02,0.026,146.470,2.044803,20592.0,None,POINT (3.42881 44.18009),eauf,EAUF-O3194010,23023208.0
12043,La Charente à Jarnac et à Mainxe,NaN,True,45.678730,-0.178207,2026-02-03 18:34:44.201628,2002-01-01,2026-02-02,3.427,480.000,48.343272,8615.0,None,POINT (-0.17821 45.67873),eauf,EAUF-R3070010,23020675.0


In [15]:
merit_dir = datasets / 'MERIT_Basins/MERIT_Hydro_v07_Basins_v01_bugfix/pfaf_level_01'
merit_gdfs = []
for b in tqdm(range(1,10), desc="Loading MERIT files"):
    filename = merit_dir / f"riv_pfaf_{b}_MERIT_Hydro_v07_Basins_v01_bugfix1.shp"
    merit_gdfs.append(gpd.read_file(filename))
print("Concatenating MERIT dataframes")
merit = gpd.GeoDataFrame(pd.concat(merit_gdfs))
merit.set_index('COMID', inplace=True)

Loading MERIT files: 100%|██████████| 9/9 [00:58<00:00,  6.45s/it]


Concatenating MERIT dataframes


In [16]:
from shapely.ops import nearest_points
from shapely.geometry import Point, LineString

def orient_line_upstream_downstream(comid):
    """
    Returns a LineString oriented so that the start is upstream
    and the end points downstream (towards nextdown).
    """
    line = merit.loc[comid, 'geometry']
    nextdown = merit.loc[comid, 'NextDownID']
    
    # If no downstream, just return as-is
    if nextdown not in merit.index or line is None or line.is_empty:
        return line

    ds_line = merit.loc[nextdown, 'geometry']
    if ds_line is None or ds_line.is_empty:
        return line
    
    # Convert start/end coords to Points
    start_pt = Point(line.coords[0])
    end_pt = Point(line.coords[-1])
    ds_start_pt = Point(ds_line.coords[0])
    
    # Compare distances to downstream start
    if start_pt.distance(ds_start_pt) < end_pt.distance(ds_start_pt):
        # First point is downstream, so reverse
        return LineString(line.coords[::-1])
    
    return line

def get_nearest_point_with_frac(row, min_frac=0.01, max_frac=0.99):
    comid = int(row["COMID"])
    line = orient_line_upstream_downstream(comid)
    if line is None or line.is_empty:
        return None, None  # return None for both geometry and fraction
    
    site_point = row.geometry
    dist = line.project(site_point)
    total_length = line.length
    if total_length == 0:
        return None, None
    
    frac = dist / total_length
    frac = max(min_frac, min(max_frac, frac))
    
    snapped_point = line.interpolate(frac * total_length)
    return snapped_point, frac

snapped_sites = matched_sites.copy()

results = snapped_sites.apply(get_nearest_point_with_frac, axis=1)
# Split into geometry and fraction
snapped_sites['geometry'] = results.map(lambda x: x[0])
snapped_sites['position'] = results.map(lambda x: x[1])

In [17]:
import networkx as nx

# build a MERIT digraph
G = nx.DiGraph()
G.add_edges_from(merit.loc[merit.NextDownID != 0, "NextDownID"].items())

In [18]:
# Lookup COMID -> geometry
comid_to_geom = merit.geometry.to_dict()

# Furthest downstream site (largest position) per COMID
downstream_site_per_comid = (
    snapped_sites.sort_values("position", ascending=False)
    .groupby("COMID", group_keys=False)
    .tail(1)
    .set_index("COMID")
)

# Mapping terminal site on each COMID
comid_to_terminal_site = downstream_site_per_comid["site_id"].to_dict()

In [20]:
# dict{COMID: NextDownID}
comid_to_down = merit["NextDownID"].to_dict()

def find_terminal_site_for_comid(comid):
    visited = set()
    while comid and comid not in visited:
        visited.add(comid)
        # If this COMID has a site, that's our terminal
        if comid in comid_to_terminal_site:
            return comid_to_terminal_site[comid]
        # move downstream
        comid = comid_to_down.get(comid, 0)
        if comid == 0:
            return None
    return None

snapped_sites["terminal_site_id"] = snapped_sites["COMID"].apply(find_terminal_site_for_comid)


In [21]:
# lookup: COMID → site_id (the furthest downstream one)
comid_to_site = downstream_site_per_comid["site_id"].to_dict()

comid_to_down = merit["NextDownID"].to_dict()

def find_terminal_site(comid):
    visited = set()
    current = comid
    terminal = None

    while current not in visited and current != 0:
        visited.add(current)

        # move downstream
        next_down = comid_to_down.get(current, 0)

        # if next reach has a site, that’s downstream gauge candidate
        if next_down in comid_to_site:
            terminal = comid_to_site[next_down]
            # but we keep going—maybe there's another gauged site further down
            current = next_down
            continue

        # if next_down == 0, we’ve reached river mouth or no further reach
        if next_down == 0:
            break

        current = next_down

    # if we found any downstream gauge, return its site_id
    if terminal:
        return terminal
        
    # else this site is terminal for its network
    return comid_to_site.get(comid)

snapped_sites["terminal_site_id"] = snapped_sites["COMID"].apply(find_terminal_site)

In [22]:
snapped_sites['site_id'].value_counts()

site_id
USGS-06935965    2
USGS-15200280    2
ECCC-04EA001     1
ECCC-06FA001     1
ECCC-08NJ026     1
                ..
USGS-14202000    1
USGS-03171000    1
USGS-11261500    1
USGS-09177000    1
USGS-02168810    1
Name: count, Length: 12043, dtype: int64

In [23]:
snapped_sites.drop_duplicates(subset='site_id', inplace=True)

In [24]:
snapped_sites['terminal_site_id'].value_counts()

terminal_site_id
USGS-07374000     1986
ECCC-02OA016       592
ABOM-213983010     488
USGS-14246900      461
EAUF-M5300010      311
                  ... 
EAUF-N3304130        1
EAUF-Y5444010        1
EAUF-E5108212        1
EAUF-12120001        1
EAUF-N2024010        1
Name: count, Length: 1471, dtype: int64

In [25]:
snapped_sites['COMID'] = snapped_sites['COMID'].astype(int)

In [19]:
# Save

# snapped_sites.to_parquet(metadata_dir / 'gauges.parquet')

In [27]:
outlets = snapped_sites.copy()
outlets.rename(columns={'site_id':'id', 'longitude':'lng', 'latitude':'lat', 'terminal_site_id':'outlet_id'}, inplace=True)

outlets = outlets[['id','lng','lat','position','COMID','outlet_id']]
# outlets.to_csv(metadata_dir / 'all_gauge_sites.csv', index=False)

In [28]:
snapped_sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider,site_id,COMID,position,terminal_site_id
0,Bradford On Avon,NaN,True,51.342781,-2.266665,2026-02-03 18:09:26.056908,2005-05-19,2026-02-02,0.422,212.344,11.605689,7563.0,{'guid': '585e49e0-7070-4d85-9c52-ccd18a5a1e65'},POINT (-2.26653 51.34264),ukea,UKEA-530176,23009540,0.635818,UKEA-530188
1,Tavistock Abbey Bridge,NaN,True,50.550625,-4.142658,2026-02-03 18:15:23.871451,1991-07-25,2026-02-01,0.230,70.092,4.410262,12608.0,{'guid': '603fcc73-a9aa-4831-bf66-6a1a2b0a04d7'},POINT (-4.14227 50.55023),ukea,UKEA-SX47F022,23012184,0.609598,UKEA-SX46F003
2,Izaak Walton,NaN,True,53.056088,-1.781900,2026-02-03 18:13:27.803203,1969-07-01,2026-02-01,0.274,26.324,1.984309,20666.0,{'guid': '73ee60b4-ef40-4579-a060-db2312ea3ce1'},POINT (-1.78166 53.05584),ukea,UKEA-4046,23003236,0.957816,UKEA-4022
3,Duddon Hall,NaN,True,54.295504,-3.237944,2026-02-03 18:06:24.406436,1976-04-01,2026-02-01,0.147,161.898,5.430623,18200.0,{'guid': 'bdd1ae70-1a37-4d43-9e04-bd378c327086'},POINT (-3.2375 54.2955),ukea,UKEA-740102,23004260,0.225076,UKEA-740102
4,Stanhope,NaN,True,54.746300,-2.026897,2026-02-03 18:15:08.786619,1958-10-01,2026-02-01,0.000,138.506,3.978364,24592.0,{'guid': 'b29c481a-5012-40f5-bb0c-f9370be34975'},POINT (-2.0269 54.74667),ukea,UKEA-024003,23001202,0.320825,UKEA-024009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12040,Le Woigot à Briey,NaN,True,49.246953,5.933378,2026-02-03 18:41:00.753027,1967-06-15,2026-02-02,0.003,22.034,1.271736,21376.0,None,POINT (5.93417 49.24695),eauf,EAUF-A8322010,23015393,0.287475,EAUF-A8500610
12041,La Lézarde à Montivilliers,NaN,True,49.543395,0.186444,2026-02-03 18:28:48.638475,2003-03-26,2026-02-02,0.590,7.824,1.080216,8349.0,None,POINT (0.18659 49.54325),eauf,EAUF-H7330410,23016883,0.759352,EAUF-H7330410
12042,La Jonte à Meyrueis [aval],NaN,True,44.180086,3.428805,2026-02-03 18:08:48.100035,1969-09-18,2026-02-02,0.026,146.470,2.044803,20592.0,None,POINT (3.42881 44.18),eauf,EAUF-O3194010,23023208,0.104061,EAUF-O9190010
12043,La Charente à Jarnac et à Mainxe,NaN,True,45.678730,-0.178207,2026-02-03 18:34:44.201628,2002-01-01,2026-02-02,3.427,480.000,48.343272,8615.0,None,POINT (-0.17805 45.67889),eauf,EAUF-R3070010,23020675,0.771612,EAUF-R5200010


In [29]:
outlets

,id,lng,lat,position,COMID,outlet_id
0,UKEA-530176,-2.266665,51.342781,0.635818,23009540,UKEA-530188
1,UKEA-SX47F022,-4.142658,50.550625,0.609598,23012184,UKEA-SX46F003
2,UKEA-4046,-1.781900,53.056088,0.957816,23003236,UKEA-4022
3,UKEA-740102,-3.237944,54.295504,0.225076,23004260,UKEA-740102
4,UKEA-024003,-2.026897,54.746300,0.320825,23001202,UKEA-024009
...,...,...,...,...,...,...
12040,EAUF-A8322010,5.933378,49.246953,0.287475,23015393,EAUF-A8500610
12041,EAUF-H7330410,0.186444,49.543395,0.759352,23016883,EAUF-H7330410
12042,EAUF-O3194010,3.428805,44.180086,0.104061,23023208,EAUF-O9190010
12043,EAUF-R3070010,-0.178207,45.678730,0.771612,23020675,EAUF-R5200010
